# 03 — Track-2 production: stance records for every legislator turn

**Purpose.** Run the stance extraction over all legislator turns in the SST
sample — the measurement workhorse of Stage 3. One row per turn, carrying the
machine-validated records (sentence indices, neutral-noun-phrase topic,
support/concern) that taxonomy induction and every panel downstream consume.

**Prompt.** `stance-extraction-0.3-dev`, run at scale on the researcher's
decision of 2026-08-22 after three dev iterations closed every observed
failure class (appendix contamination, token-cap truncation, stance-target
inversion, rhetorical-enumeration over-extraction, adjacent duplicates,
markdown fences). Every call logs the prompt version, so these records
remain distinguishable from any later wording.

**Sharded and resumable.** `iloc[SHARD::N_SHARDS]` on the stored turn order;
a turn is done when its row is in the shard parquet, which is rewritten
every 200 turns — a task killed at the wall clock loses at most a few
minutes, and a re-submission skips finished turns. Call logs are one JSONL
per shard (a file per turn would be 221k files of inode abuse on shared
storage).

**Inputs.** `interim/topics/track2/member_turns.parquet` — from `01`
(paragraph-wise segmentation + cleanup rules).

**Outputs.** `interim/topics/track2/shards/records_{shard}.parquet` — one row
per turn with serialized records; `status/extract_{shard}.jsonl` — every
call. A later assembly step explodes the shards into `stance_records.parquet`
(one row per record) once all shards finish.

**Execute via** `jobs/extract_stance.sbatch` (5-task GPU array).

## Setup and parameters

Environment-variable parameterization, so all shards run the same file and
each records which slice it ran in its own output.

In [ ]:
import hashlib
import json
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, "../lib")

from paths import HF_CACHE, topics_run_dir

HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
for line in (Path(".env").read_text().splitlines() if Path(".env").exists() else []):
    if line.startswith("HF_TOKEN="):
        os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip()

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import stance
import summarize as S

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
SHARD = int(os.environ.get("SHARD", 0))
N_SHARDS = int(os.environ.get("N_SHARDS", 5))
MAX_NEW_TOKENS = 1024

TRACK2 = topics_run_dir("track2")
SHARD_DIR = TRACK2 / "shards"
SHARD_DIR.mkdir(exist_ok=True)
SHARD_FILE = SHARD_DIR / f"records_{SHARD:02d}.parquet"
LOG_FILE = TRACK2 / "status" / f"extract_{SHARD:02d}.jsonl"

print(f"model   : {MODEL}")
print(f"prompt  : {stance.PROMPT_VERSION}")
print(f"shard   : {SHARD} of {N_SHARDS}")
print(f"window  : {stance.WINDOW_SENTENCES} sentences, cap {MAX_NEW_TOKENS}")
print(f"sampling: {S.GENERATION}")
print(f"gpu     : {torch.cuda.get_device_name(0)}")

## Step 1 — This shard's worklist

The stride partitions the stored turn order, so shard membership is a
property of the file, not the run — a re-submission selects the same turns.
Turns emptied by the cleanup rules are excluded here (nothing to extract),
and finished turns are read back from the shard parquet.

In [ ]:
turns = pd.read_parquet(TRACK2 / "member_turns.parquet")
work = turns.iloc[SHARD::N_SHARDS]
work = work.loc[work["n_sentences"] > 0].reset_index(drop=True)

rows = []
if SHARD_FILE.exists():
    rows = pd.read_parquet(SHARD_FILE).to_dict("records")
done = {(r["packageId"], r["turn_idx"]) for r in rows}
todo = work.loc[[
    (p, t) not in done
    for p, t in zip(work["packageId"], work["turn_idx"])]].reset_index(drop=True)

print(f"shard turns : {len(work):,} ({work['n_sentences'].sum():,} sentences)")
print(f"  done      : {len(done):,}")
print(f"  to extract: {len(todo):,}")

## Step 2 — Load the model

In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s — "
      f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")

## Step 3 — Extract

Per turn: seed from the turn key (bit-reproducible per turn, order-free),
one call per window with turn-global numbering, machine validation, then the
adjacent-duplicate merge. The shard parquet is rewritten every 200 turns;
the call log is appended and flushed per call, so nothing that happened is
ever only in memory.

In [ ]:
started = time.time()
n_calls_session = 0
for n, row in enumerate(todo.itertuples(), 1):
    key = f"{row.packageId}:{row.turn_idx}"
    torch.manual_seed(int.from_bytes(
        hashlib.sha256(key.encode()).digest()[:4], "big"))
    speaker = f"{row.honorific or ''} {row.surname or ''}".strip() or "Unidentified"
    log = S.CallLog(LOG_FILE, key, MODEL,
                    prompt_version=stance.PROMPT_VERSION)

    records, errors = [], []
    for w_index, (w_start, w_sents) in enumerate(
            stance.windows(list(row.sentences))):
        user = stance.render_user(w_sents, speaker, start=w_start)
        t0 = time.time()
        output, hit_cap = S._generate(model, tokenizer, stance.SYSTEM, user,
                                       max_new_tokens=MAX_NEW_TOKENS)
        log.write("extract", 0, w_index, 1, user, output, tokenizer,
                  time.time() - t0)
        n_calls_session += 1
        w_records, w_errors = stance.parse_records(output, len(w_sents),
                                                    start=w_start)
        if hit_cap:
            w_errors.append("hit_token_cap")
        records.extend(w_records)
        errors.extend(w_errors)
    records, n_merged = stance.merge_adjacent_records(records)

    rows.append({
        "packageId": row.packageId, "turn_idx": int(row.turn_idx),
        "year": int(row.year), "chamber": row.chamber,
        "bioGuideId": row.bioGuideId, "surname": row.surname,
        "honorific": row.honorific, "is_chair": bool(row.is_chair),
        "n_sentences": int(row.n_sentences),
        "prompt_version": stance.PROMPT_VERSION, "model": MODEL,
        "n_records": len(records), "n_merged": n_merged,
        "records_json": json.dumps(records),
        "errors": ";".join(errors),
    })
    if n % 200 == 0 or n == len(todo):
        pd.DataFrame(rows).to_parquet(SHARD_FILE, compression="zstd", index=False)
        rate = n / (time.time() - started)
        print(f"  {n:>6,}/{len(todo):,}  {rate:.2f} turns/s  "
              f"eta {(len(todo) - n) / rate / 3600:.1f} h")

pd.DataFrame(rows).to_parquet(SHARD_FILE, compression="zstd", index=False)
print(f"\nshard {SHARD}: {len(rows):,} turns, {n_calls_session:,} calls this "
      f"session, {(time.time() - started) / 3600:.2f} h")

## Verification (this shard)

Alarm bands from the dev runs: structural-error rate at or below the dev
level (~3%), records-per-turn in a plausible band, both stances present.
Out-of-band numbers mean stop and inspect before assembly, not proceed.

In [ ]:
back = pd.read_parquet(SHARD_FILE)
assert not back.duplicated(["packageId", "turn_idx"]).any(), "turn done twice"

err_rate = (back["errors"] != "").mean()
rec_per_turn = back["n_records"].mean()
all_records = [r for js in back["records_json"] for r in json.loads(js)]
stances = pd.Series([r["stance"] for r in all_records]).value_counts()
lint = sum(1 for r in all_records if r.get("topic_lint"))

print(f"1) turns          : {len(back):,} (no duplicates)")
print(f"2) error rate     : {err_rate:.1%} "
      f"[{'OK' if err_rate < 0.05 else 'OUT OF BAND — inspect'}]")
print(f"3) records        : {len(all_records):,} "
      f"({rec_per_turn:.2f}/turn) "
      f"[{'OK' if 0.2 <= rec_per_turn <= 3.0 else 'OUT OF BAND — inspect'}]")
print(f"4) stance mix     : {stances.to_dict()}")
print(f"5) topic lint     : {lint:,} flagged "
      f"({lint / max(len(all_records), 1):.1%}) — review queue, not removal")
print(f"6) empty-list     : {(back['n_records'] == 0).mean():.0%} of turns")
print(f"7) prompt version : {back['prompt_version'].unique().tolist()}")